# DTL-5G Code Sample 3 — DTL1 BiLSTM Transfer Learning
## 5G-NIDD, Objective 1, iteration 1 (`random_state = 42`)

DTL1 freezes the first six of nine layers and fine-tunes the final three layers
on the 5G-NIDD target domain.

By default, this notebook loads the source checkpoint created by Notebook 1.
Set `DTL5G_SOURCE_MODEL` to use another source checkpoint.


## 1. Imports and configuration

In [ ]:
from pathlib import Path
import os
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from tensorflow import keras
from tensorflow.keras import regularizers
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import Activation, Bidirectional, Dense, Dropout, LSTM
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
)

OBJECTIVE1_RANDOM_STATES = [42, 142, 12, 4, 80]
ITERATION_SEED = OBJECTIVE1_RANDOM_STATES[0]  # Objective 1, iteration 1

DATA_ROOT = Path(os.getenv("NIDD_DATA_ROOT", "./data/5G-NIDD")).expanduser()
ATTACK_DIR = DATA_ROOT / "Attack"
BENIGN_DIR = DATA_ROOT / "Benign"

# Default: checkpoint created by Notebook 1.
# Set DTL5G_SOURCE_MODEL to another selected source checkpoint when needed.
SOURCE_MODEL_PATH = Path(
    os.getenv(
        "DTL5G_SOURCE_MODEL",
        "./artifacts/BiLSTM_VM_Obj1_1.keras",
    )
).expanduser()

ARTIFACT_DIR = Path("artifacts")
ARTIFACT_DIR.mkdir(exist_ok=True)


In [ ]:
# Reproducible model initialization, dropout masks, NumPy operations,
# and TensorFlow random operations for this iteration.
keras.utils.set_random_seed(ITERATION_SEED)

# Request deterministic TensorFlow kernels where supported.
try:
    tf.config.experimental.enable_op_determinism()
    print("TensorFlow deterministic operations enabled.")
except Exception as exc:
    print(f"Deterministic TensorFlow operations unavailable: {exc}")

## 2. Target features and 5G-NIDD data manifest

In [ ]:
FEATURES = ['Flow Duration', 'Fwd Pkt Len Std', 'ACK Flag Cnt', 'Protocol', 'Tot Fwd Pkts', 'TotLen Fwd Pkts', 'TotLen Bwd Pkts', 'Tot Bwd Pkts']
LABEL_COLUMN = "Label"

ATTACK_FILES = ['SYNflood_BS1_nogtp.pcap_Flow.csv', 'SYNflood_BS2_nogtp.pcap_Flow.csv', 'UDPflood_BS1_nogtp.pcap_Flow.csv', 'UDPflood_BS2_nogtp.pcap_Flow.csv', 'Goldeneye_BS1_nogtp.pcapng_Flow.csv', 'Goldeneye_BS2_nogtp.pcapng_Flow.csv', 'ICMPflood_BS1_nogtp.pcapng_Flow.csv', 'ICMPflood_BS2_nogtp.pcapng_Flow.csv', 'Slowloris_BS1_nogtp.pcapng_Flow.csv', 'Slowloris_BS2_nogtp.pcapng_Flow.csv', 'UDPScan_BS2_nogtp.pcapng_Flow.csv', 'UDPScan_BS1_nogtp.pcapng_Flow.csv', 'SYNscan_BS2_nogtp.pcapng_Flow.csv', 'SYNScan_BS1_nogtp.pcapng_Flow.csv', 'TCPConnect_BS1_nogtp.pcapng_Flow.csv', 'TCPConnect_BS2_nogtp.pcapng_Flow.csv', 'Torshammer_BS1_nogtp.pcapng_Flow.csv', 'Torshammer_BS2_nogtp.pcapng_Flow.csv']
BENIGN_FILES = ['Goldeneye_BS1_nogtp.pcap_Flow.csv', 'Goldeneye_BS2_nogtp.pcap_Flow.csv', 'ICMPflood_BS1_nogtp.pcap_Flow.csv', 'ICMPflood_BS2_nogtp.pcap_Flow.csv', 'Slowloris_BS1_nogtp.pcap_Flow.csv', 'Slowloris_BS2_nogtp.pcap_Flow.csv', 'SYNflood_BS1_nogtp.pcap_Flow.csv', 'SYNflood_BS2_nogtp.pcap_Flow.csv', 'SYNScan_BS1_nogtp.pcap_Flow.csv', 'SYNscan_BS2_nogtp.pcap_Flow.csv', 'UDPflood_BS1_nogtp.pcap_Flow.csv', 'UDPflood_BS2_nogtp.pcap_Flow.csv', 'SSH_BS1_nogtp.pcap_Flow.csv', 'SSH_BS2_nogtp.pcap_Flow.csv', 'TCPConnect_BS1_nogtp.pcap_Flow.csv', 'TCPConnect_BS2_nogtp.pcap_Flow.csv', 'Torshammer_BS1_nogtp.pcap_Flow.csv', 'Torshammer_BS2_nogtp.pcap_Flow.csv', 'UDPScan_BS1_nogtp.pcap_Flow.csv', 'UDPScan_BS2_nogtp.pcap_Flow.csv']

EXPECTED_BENIGN_COUNT = 15_212
EXPECTED_ATTACK_COUNT = 315_665

def assert_unique_manifest(name, filenames):
    duplicates = sorted({f for f in filenames if filenames.count(f) > 1})
    if duplicates:
        raise ValueError(f"{name} contains duplicate filenames: {duplicates}")

assert_unique_manifest("BENIGN_FILES", BENIGN_FILES)
assert_unique_manifest("ATTACK_FILES", ATTACK_FILES)


## 3. Load, clean and balance 5G-NIDD

In [ ]:
def load_group(directory, filenames, label):
    paths = [directory / name for name in filenames]
    missing = [str(path) for path in paths if not path.exists()]
    if missing:
        raise FileNotFoundError(
            "Missing 5G-NIDD CSV files. Set NIDD_DATA_ROOT.\n"
            + "\n".join(missing[:10])
        )

    frame = pd.concat((pd.read_csv(path) for path in paths), ignore_index=True)
    frame[LABEL_COLUMN] = label
    return frame[FEATURES + [LABEL_COLUMN]]

attack_df = load_group(ATTACK_DIR, ATTACK_FILES, "attack")
benign_df = load_group(BENIGN_DIR, BENIGN_FILES, "Benign")

assert len(benign_df) == EXPECTED_BENIGN_COUNT
assert len(attack_df) == EXPECTED_ATTACK_COUNT

attack_df = attack_df.replace([np.inf, -np.inf], np.nan)
benign_df = benign_df.replace([np.inf, -np.inf], np.nan)

def validate_selected_features(frame, name):
    values = frame[FEATURES].to_numpy(dtype=np.float64)
    invalid = ~np.isfinite(values)

    if invalid.any():
        rows, cols = np.where(invalid)
        examples = [
            (int(r), FEATURES[int(c)])
            for r, c in zip(rows[:10], cols[:10])
        ]
        raise ValueError(
            f"{name} contains {invalid.sum()} non-finite selected-feature values. "
            f"Examples: {examples}"
        )

    print(f"{name}: selected features are finite.")

validate_selected_features(attack_df, "5G-NIDD attack traffic")
validate_selected_features(benign_df, "5G-NIDD benign traffic")


attack_df = attack_df.sample(
    n=len(benign_df),
    random_state=ITERATION_SEED,
    replace=False,
)

## 4. Feature matrix, stratified split and StandardScaler

In [ ]:
X_benign = benign_df[FEATURES].to_numpy(dtype=np.float32)
X_attack = attack_df[FEATURES].to_numpy(dtype=np.float32)

X = np.concatenate((X_benign, X_attack), axis=0)
y = np.concatenate((
    np.ones(len(benign_df), dtype=np.int32),
    np.zeros(len(attack_df), dtype=np.int32),
))

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=ITERATION_SEED,
    stratify=y,
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

X_train = np.expand_dims(X_train, axis=-1)
X_test = np.expand_dims(X_test, axis=-1)

assert X.shape == (30_424, 8)
assert len(X_test) == 6_085

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

## 5. Load and validate the source checkpoint

In [ ]:
if not SOURCE_MODEL_PATH.exists():
    raise FileNotFoundError(
        "Source checkpoint not found. Run Notebook 1 first or set "
        "DTL5G_SOURCE_MODEL to an existing source-model checkpoint."
    )

model = keras.models.load_model(SOURCE_MODEL_PATH)

EXPECTED_LAYERS = 9
EXPECTED_TOTAL_PARAMS = 99_970

assert len(model.layers) == EXPECTED_LAYERS, (
    f"Expected {EXPECTED_LAYERS} layers, found {len(model.layers)}."
)
assert model.count_params() == EXPECTED_TOTAL_PARAMS, (
    f"Expected {EXPECTED_TOTAL_PARAMS:,} parameters, "
    f"found {model.count_params():,}."
)

model.summary()


## 6. DTL1: retain only the final 33% of layers as trainable

In [ ]:
# 9 layers -> final 3 layers = 33.3%.
for layer in model.layers:
    layer.trainable = False

for layer in model.layers[-3:]:
    layer.trainable = True

# Recompile after changing trainability.
model.compile(
    loss="binary_crossentropy",
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    metrics=["accuracy"],
)

trainable_params = int(
    sum(np.prod(weight.shape) for weight in model.trainable_weights)
)
frozen_params = model.count_params() - trainable_params

print("Layer status:")
for index, layer in enumerate(model.layers, start=1):
    status = "trainable" if layer.trainable else "frozen"
    print(f"{index}: {layer.name:<20} {status}")

print("Total parameters:", f"{model.count_params():,}")
print("DTL1 trainable:", f"{trainable_params:,}")
print("DTL1 frozen:", f"{frozen_params:,}")

# Layers 7–9 are Dense(128), Dropout, Dense(2):
# 32,896 + 0 + 258 = 33,154 trainable parameters.
assert trainable_params == 33_154
assert frozen_params == 66_816

### DTL1 parameter audit

The final 3 of 9 layers are retrained:

- Dense(128): 32,896 parameters
- Dropout: 0 parameters
- Dense(2): 258 parameters

**DTL1 trainable = 33,154**  
**Frozen = 66,816**  
**Total = 99,970**

## 7. Fine-tune on 5G-NIDD

In [ ]:
y_train_one_hot = keras.utils.to_categorical(y_train.astype(float), num_classes=2)

early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=15,
    min_delta=0.0001,
)

start = time.perf_counter()
history = model.fit(
    X_train,
    y_train_one_hot,
    epochs=200,
    validation_split=0.20,
    callbacks=[early_stopping],
    verbose=1,
)
training_seconds = time.perf_counter() - start

print(f"Fine-tuning time: {training_seconds / 60:.2f} min")
print(f"Fine-tuning time/sample: {(training_seconds * 1000) / len(X_train):.4f} ms")

## 8. Save DTL1 model

In [ ]:
DTL1_MODEL_PATH = ARTIFACT_DIR / "DTL1_BiLSTM_5GNIDD_Obj1_1.keras"
model.save(DTL1_MODEL_PATH)
print("Saved:", DTL1_MODEL_PATH)

## 9. Evaluate

In [ ]:
def evaluate_attack_positive(model, X_test, y_test):
    """Evaluate using the evaluation convention: attack is the positive class."""
    probabilities = model.predict(X_test, verbose=0)
    predictions = probabilities.argmax(axis=1)

    metrics_out = {
        "accuracy": accuracy_score(y_test, predictions),
        "recall": recall_score(y_test, predictions, pos_label=0, zero_division=0),
        "precision": precision_score(y_test, predictions, pos_label=0, zero_division=0),
        "f1": f1_score(y_test, predictions, pos_label=0, zero_division=0),
        "auc": roc_auc_score((y_test == 0).astype(int), probabilities[:, 0]),
    }

    for key, value in metrics_out.items():
        print(f"{key.capitalize():>9}: {value:.4f}")

    return probabilities, predictions, metrics_out

probabilities, predictions, result = evaluate_attack_positive(
    model, X_test, y_test
)

## 10. Inference timing

In [ ]:
def benchmark(device):
    with tf.device(device):
        start = time.perf_counter()
        _ = model.predict(X_test, verbose=0)
        elapsed = time.perf_counter() - start
    return elapsed, elapsed * 1000 / len(X_test)

_, cpu_each = benchmark("/CPU:0")
print(f"CPU: {cpu_each:.4f} ms/sample")

if tf.config.list_physical_devices("GPU"):
    _, gpu_each = benchmark("/GPU:0")
    print(f"GPU: {gpu_each:.4f} ms/sample")

## 11. Confusion matrix

In [ ]:
cm = confusion_matrix(y_test, predictions, labels=[0, 1])
cm_df = pd.DataFrame(
    cm,
    index=["Attack", "Benign"],
    columns=["Attack", "Benign"],
)

plt.figure(figsize=(6, 4))
sns.heatmap(cm_df, annot=True, fmt="d", cbar=False)
plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.title("DTL1 BiLSTM — Objective 1, iteration 1")
plt.tight_layout()
plt.show()

## Five-run aggregation

This notebook contains one target-domain Objective-1 iteration. The complete
experiment uses the fixed seeds `42, 142, 12, 4, 80`, and final metrics are
obtained by averaging across the five runs.
